In [0]:
from pyspark.sql.functions import *

# ==========================================
# READ BRONZE TABLE
# ==========================================

csv_df = spark.table("workauto.bronze.csv_raw")

# ==========================================
# REMOVE DUPLICATES USING BUSINESS KEY
# ==========================================

csv_df = csv_df.dropDuplicates(["order_id"])

# ==========================================
# CLEANING
# ==========================================

csv_df = (
    csv_df
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("product", trim(col("product")))
    .withColumn("email", trim(col("email")))
)

# STANDARDIZE TEXT
csv_df = (
    csv_df
    .withColumn("customer_name", initcap(col("customer_name")))
    .withColumn("city", initcap(col("city")))
    .withColumn("product", initcap(col("product")))
)

# HANDLE NULLS
csv_df = csv_df.fillna({
    "customer_name": "Unknown",
    "product": "Unknown Product"
})

# REMOVE INVALID DATA
csv_df = csv_df.filter(col("quantity") > 0)

csv_df = csv_df.filter(col("price").isNotNull())

# EMAIL VALIDATION
csv_df = csv_df.withColumn(
    "email_status",
    when(
        col("email").rlike(
            r"^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\.[a-zA-Z]{2,}$"
        ),
        "Valid"
    ).otherwise("Invalid")
)

# WORKFLOW LOGIC
csv_df = csv_df.withColumn(
    "workflow_status",
    when(
        col("email_status") == "Invalid",
        "Needs Review"
    ).otherwise("Ready")
)

# PROCESS TIMESTAMP
csv_df = csv_df.withColumn(
    "processed_time",
    current_timestamp()
)

# SHOW DATA
csv_df.show(truncate=False)

# ==========================================
# SAVE SILVER TABLE
# ==========================================

(
    csv_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workauto.silver.csv_clean")
)

print("CSV Silver Table Refreshed Successfully")